In [1]:
# 라이브러리 불러오기
from selenium import webdriver as wb # web browser
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time
import pandas as pd
import re
from tqdm import tqdm

from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager # 자동으로 드라이버 관리

# 1. Chrome 옵션 설정 (리눅스에서는 headless 모드 추천)L
chrome_options = Options()
chrome_options.add_argument('--headless') # 브라우저 창을 띄우지 않고 실행
chrome_options.add_argument('--no-sandbox') # 리눅스에서 권장되는 옵션
chrome_options.add_argument('--disable-dev-shm-usage') # 공유 메모리 사용 비활성화

In [2]:
# 문자열 전처리 함수 -> 숫자, 문자, 특수문자 제외하고 삭제
def preprocess_sentence_kr(w):
  w = w.strip()
  w = re.sub(r"[^0-9가-힣?.!,¿]+", " ", w) 
  w = w.strip() 
  return w

### 단일 경우 test

In [3]:
# 네이버 메인페이지 -> "음식물 처리기" 검색 -> 지식인 탭 클릭

driver = wb.Chrome(options=chrome_options)
driver.get('https://www.naver.com/') 

In [4]:
s = driver.find_element(By.ID, 'query')
keyword = '냉장고 정리'
s.send_keys(keyword +'\n') # \n 브라우저 입장에서는 Enter 키로 인식

In [5]:
# 블로그 탭 클릭
# in_click = driver.find_element(By.CSS_SELECTOR, '#lnb > div.lnb_group > div > div.lnb_nav_area._nav_area_root > div > div.api_flicking_wrap._conveyer_root > div:nth-child(1) > a')
in_click = driver.find_element(By.CSS_SELECTOR, '#lnb > div.lnb_group > div > div.lnb_nav_area._nav_area_root > div > div.api_flicking_wrap._conveyer_root > div:nth-child(2) > a')
in_click.click()

- 기간 설정을 통하여 6개월 이내의 데이터만 노출

In [6]:
# 기간값을 가진 요소 추출 -> 클릭
option_btn = driver.find_element(By.CSS_SELECTOR, '#snb > div.mod_group_option_filter._search_option_simple_wrap > div > div.option_filter > a')
option_btn.click()
time.sleep(1)

In [7]:
# 기간값을 가진 요소 추출 -> 클릭
option_btn = driver.find_element(By.CSS_SELECTOR, '#snb > div.mod_group_option_sort._search_option_detail_wrap > ul > li.bx.term > div > div.option > a:nth-child(8)')
option_btn.click()
time.sleep(1)

In [8]:
# 스크롤 내리기 (전체화면(body)에 END 키를 전송)
# for i in range(10):
#     body = driver.find_element(By.TAG_NAME, 'body')
#     body.send_keys(Keys.END)
#     time.sleep(1)
    
    
# 웹페이지의 높이 값을 활용한 스크롤 내리기!
# 현재 페이지의 총 높이 추출 (이전페이지 높이, old_height)
old_height = driver.execute_script('return document.documentElement.scrollHeight')

# 반복문을 통하여 스크롤 -> 새로운 페이지와 높이를 비교하여 끝까지 내리기
while True:
    body = driver.find_element(By.TAG_NAME, 'body')
    body.send_keys(Keys.END)
    time.sleep(1)
    
    new_height = driver.execute_script('return document.documentElement.scrollHeight')
    print("new_height: ", new_height)
    if new_height == old_height:
        print('Done')
        break
    else:
        old_height = new_height

new_height:  11831
new_height:  17029
new_height:  22227
new_height:  27425
new_height:  32623
new_height:  37821
new_height:  43019
new_height:  48217
new_height:  53415
new_height:  58613
new_height:  63811
new_height:  69009
new_height:  74207
new_height:  79405
new_height:  84603
new_height:  89801
new_height:  94997
new_height:  100195
new_height:  105393
new_height:  110591
new_height:  115789
new_height:  120985
new_height:  126183
new_height:  131381
new_height:  136579
new_height:  136579
Done


In [9]:
# 블로그 게시글 url 수집
urls = driver.find_elements(By.CSS_SELECTOR, 'a.fender-ui_228e3bd1.vnDTNtnoko3GR0aJilUy')

# href 데이터만 리스트에 저장 (href_list)
# 요소.get_attribute('href')
href_list = [url.get_attribute('href') for url in urls]
title_list = [url.text for url in urls]

print(len(href_list))
print(href_list[0])

print(len(title_list))
print(title_list[0])

780
https://blog.naver.com/hyehun15/224079118913
780
음식물처리기 미닉스 더 플렌더 MAX 3주 사용 냉장고 정리도


In [10]:
# 1개 url에 대해서 우선 진행

driver.get(href_list[0])
time.sleep(1)

driver.switch_to.frame('mainFrame')
contents = driver.find_element(By.CSS_SELECTOR, 'div.se-main-container')
contents.text

driver.quit()

### Macro

In [11]:
# 1. 확보된 url 주소를 통해 반복적으로 요청
# 2. 게시글 내 작성된 리뷰를 추출하여 텍스트 파일로 저장
# 파일명: 네이버 지식인 리뷰 데이터.txt

driver = wb.Chrome(options=chrome_options)
time.sleep(2)

f = open(f'./results/네이버_블로그_{keyword}.txt', 'w', encoding='utf-8')
contents_lst = []

for href in tqdm(href_list):
    try:
        driver.get(href)
        time.sleep(1)
        driver.switch_to.frame('mainFrame')
        time.sleep(1)
        
        contents = driver.find_element(By.CSS_SELECTOR, 'div.se-main-container')
        contents_preprocessed = preprocess_sentence_kr(contents.text)
        f.write(contents_preprocessed)
        contents_lst.append(contents_preprocessed)
    except:
        continue
    
driver.quit()
    
f.close()

100%|██████████| 780/780 [43:45<00:00,  3.37s/it]


In [12]:
# dataframe 저장
df = pd.DataFrame([contents_lst]).T
df.columns = ['content']
df.to_excel(f'./results/네이버_블로그_{keyword}.xlsx', index=False)

In [13]:
df

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


,content
0,본 리뷰는 업체로부터 제품을 지원받아 직접 사용하고 작성된 솔직한 후기입니다. 냉장...
1,트라이탄 밀폐용기 추천 냉장고정리팁 엔에프락 투명 냉장고수납용기 글. 사진. 영상 ...
2,냉장고 정리팁 이제 그만 썩히자! 버리는 돈 막는 식재료 관리 시스템 안녕하세요! ...
3,"데일리앤쿡 프레실버 세균 번식 제로 밀폐 항균 용기 은은한 안심, 오래가는 신선함 ..."
4,주방을 아무리 치워도 금세 어수선해지는 것은 바로 냉장고 속 정리 부족 때문일 수 ...
...,...
771,"실리만 밀폐용기로 쉽게 냉장고 정리 참깨방앗간 안녕하세요, 참깨입니다 요즘은 저희집..."
772,안녕하세요! 30대 취준생으로써 시간이 많이 남아도는 승쨩이에요 그래서 이참에 이지...
773,반가워요. 푸드 인플루언서 해피잇입니다. 오늘은 냉장고 정리 음식으로 만드는 간단한...
774,야채 오랫동안 보관하는 법 냉장고 정리 팬트리 정리 쉽게 하기 오키퍼 원터치 밀폐용...


- 수집된 데이터 시각화
- 워드클라우드
    - kiwi 라이브러리 활용하여 토큰화
    - 빈도수 상위 100개 단어 활용하여 워드클라우드 생성

In [14]:
# 라이브러리 불러오기
import numpy as np
from kiwipiepy import Kiwi # 형태소 분석기
from collections import Counter # 단어가 나온 횟수를 셈
from wordcloud import WordCloud # 워드클라우드 생성 도구
import matplotlib.pyplot as plt # 시각화 도구
from PIL import Image

In [15]:
f = open(f'./results/네이버_블로그_{keyword}.txt', 'r', encoding='utf-8')
review = f.read()
f.close()
print(len(review))

1439320


In [16]:
# 도구 객체생성
kiwi = Kiwi()

# 토큰화
token = kiwi.tokenize(review)
token

# form: 토큰화 결과
# tag: 품사 태크
# nnp: 고유명사; nng: 일반명사

[Token(form='본', tag='MM', start=0, len=1),
 Token(form='리뷰', tag='NNG', start=2, len=2),
 Token(form='는', tag='JX', start=4, len=1),
 Token(form='업체', tag='NNG', start=6, len=2),
 Token(form='로부터', tag='JKB', start=8, len=3),
 Token(form='제품', tag='NNG', start=12, len=2),
 Token(form='을', tag='JKO', start=14, len=1),
 Token(form='지원', tag='NNG', start=16, len=2),
 Token(form='받', tag='VV-R', start=18, len=1),
 Token(form='어', tag='EC', start=19, len=1),
 Token(form='직접', tag='MAG', start=21, len=2),
 Token(form='사용', tag='NNG', start=24, len=2),
 Token(form='하', tag='XSV', start=26, len=1),
 Token(form='고', tag='EC', start=27, len=1),
 Token(form='작성', tag='NNG', start=29, len=2),
 Token(form='되', tag='XSV', start=31, len=1),
 Token(form='ᆫ', tag='ETM', start=31, len=1),
 Token(form='솔직', tag='XR', start=33, len=2),
 Token(form='하', tag='XSA', start=35, len=1),
 Token(form='ᆫ', tag='ETM', start=35, len=1),
 Token(form='후기', tag='NNG', start=37, len=2),
 Token(form='이', tag='VCP', star

In [17]:
# 일반명사만 추출 (NNG)
# 리스트명 = [실행문 반복문 조건]
# nng_token = [t.form for t in token if t.tag=='NNG']
nng_token = [t.form for t in token if t.tag=='NNG' or t.tag=='NNP']

print(len(nng_token))
print(nng_token)

228850
['리뷰', '업체', '제품', '지원', '사용', '작성', '후기', '냉장고', '정리', '깔끔', '공간', '수납', '분쇄', '음식물', '처리', '사용', '후기', '압도', '성능', '음식물', '처리', '미닉스', '더', '플렌더', '기대감', '얼마', '전', '기대', '장만', '제품', '기대', '이상', '만족', '주방', '생활', '르', '때', '제품', '대비', '시간', '단축', '사용', '업그레이드', '시간', '단축', '시간', '반', '끝', '하루', '특', '이번', '제품', '미닉스', '더', '플렌더', '음식물', '처리', '과일', '조개', '껍데기', '닭', '뼈', '생선', '뼈', '무리', '건조', '분쇄', '사용', '기간', '동안', '부담', '공간', '무리', '설치', '가능', '제품', '모던', '디자인', '주방', '공간', '인테리어', '소음', '냄새', '집', '거실', '다이닝', '공간', '사용', '음쓰', '처리', '냉장고', '정리', '이상', '냉장고', '음쓰', '집', '아파트', '빌라', '주택', '형태', '음식물', '수거', '차량', '수거', '장', '불편', '유통', '기한', '음식물', '위생', '생활', '사용', '기간', '동안', '시간', '반', '단축', '시간', '하루', '부담', '음식물', '처리', '미닉스', '더', '플렌더', '냉장고', '정리', '유통', '기한', '냉동식품', '경우', '냉동', '상태', '부피', '종량제', '봉투', '낭비', '때', '냄새', '벌레', '스트레스', '나중', '요즘', '유통', '기한', '냉동식품', '보관', '떡', '음쓰', '처리', '음식물', '처리', '사용', '냉장고', '정리', '위생', '주방', '슬림', '디자인', '컴팩트', '용량', '업그레이드'

In [18]:
# 2글자 이상의 단어만 추출
nng_token2 = [t for t in nng_token if len(t)>=2]
nng_token2

['리뷰',
 '업체',
 '제품',
 '지원',
 '사용',
 '작성',
 '후기',
 '냉장고',
 '정리',
 '깔끔',
 '공간',
 '수납',
 '분쇄',
 '음식물',
 '처리',
 '사용',
 '후기',
 '압도',
 '성능',
 '음식물',
 '처리',
 '미닉스',
 '플렌더',
 '기대감',
 '얼마',
 '기대',
 '장만',
 '제품',
 '기대',
 '이상',
 '만족',
 '주방',
 '생활',
 '제품',
 '대비',
 '시간',
 '단축',
 '사용',
 '업그레이드',
 '시간',
 '단축',
 '시간',
 '하루',
 '이번',
 '제품',
 '미닉스',
 '플렌더',
 '음식물',
 '처리',
 '과일',
 '조개',
 '껍데기',
 '생선',
 '무리',
 '건조',
 '분쇄',
 '사용',
 '기간',
 '동안',
 '부담',
 '공간',
 '무리',
 '설치',
 '가능',
 '제품',
 '모던',
 '디자인',
 '주방',
 '공간',
 '인테리어',
 '소음',
 '냄새',
 '거실',
 '다이닝',
 '공간',
 '사용',
 '음쓰',
 '처리',
 '냉장고',
 '정리',
 '이상',
 '냉장고',
 '음쓰',
 '아파트',
 '빌라',
 '주택',
 '형태',
 '음식물',
 '수거',
 '차량',
 '수거',
 '불편',
 '유통',
 '기한',
 '음식물',
 '위생',
 '생활',
 '사용',
 '기간',
 '동안',
 '시간',
 '단축',
 '시간',
 '하루',
 '부담',
 '음식물',
 '처리',
 '미닉스',
 '플렌더',
 '냉장고',
 '정리',
 '유통',
 '기한',
 '냉동식품',
 '경우',
 '냉동',
 '상태',
 '부피',
 '종량제',
 '봉투',
 '낭비',
 '냄새',
 '벌레',
 '스트레스',
 '나중',
 '요즘',
 '유통',
 '기한',
 '냉동식품',
 '보관',
 '음쓰',
 '처리',
 '음식물',
 '처리',
 '사용',
 '냉장고',
 '정리',
 '위생',

In [19]:
# 명사의 개수를 세서 많이 나오는 단어를 활용하여 워드클라욷 생성하기
counter = Counter(nng_token2)
counter

Counter({'냉장고': 9366,
         '정리': 9134,
         '용기': 7434,
         '보관': 5608,
         '사용': 4545,
         '밀폐': 3083,
         '반찬': 2033,
         '가능': 1905,
         '제품': 1694,
         '수납': 1564,
         '식재료': 1553,
         '진공': 1454,
         '음식': 1376,
         '추천': 1273,
         '주방': 1248,
         '활용': 1187,
         '냉동실': 1177,
         '사이즈': 1177,
         '실리콘': 1118,
         '공간': 1099,
         '투명': 1067,
         '뚜껑': 1057,
         '야채': 1010,
         '냉동': 986,
         '세트': 939,
         '김치': 926,
         '과일': 925,
         '지퍼백': 916,
         '냄새': 901,
         '필요': 893,
         '재료': 853,
         '트레이': 780,
         '다양': 777,
         '채소': 776,
         '포장': 775,
         '음료': 754,
         '소재': 752,
         '유지': 737,
         '걱정': 704,
         '다이소': 703,
         '박스': 675,
         '데비 마이어': 641,
         '라벨': 628,
         '소스': 627,
         '프리': 613,
         '생각': 597,
         '구성': 587,
         '구매': 587,
     

In [20]:
# 상위 100개 단어만 추출
top_100 = counter.most_common(100)
top_100

[('냉장고', 9366),
 ('정리', 9134),
 ('용기', 7434),
 ('보관', 5608),
 ('사용', 4545),
 ('밀폐', 3083),
 ('반찬', 2033),
 ('가능', 1905),
 ('제품', 1694),
 ('수납', 1564),
 ('식재료', 1553),
 ('진공', 1454),
 ('음식', 1376),
 ('추천', 1273),
 ('주방', 1248),
 ('활용', 1187),
 ('냉동실', 1177),
 ('사이즈', 1177),
 ('실리콘', 1118),
 ('공간', 1099),
 ('투명', 1067),
 ('뚜껑', 1057),
 ('야채', 1010),
 ('냉동', 986),
 ('세트', 939),
 ('김치', 926),
 ('과일', 925),
 ('지퍼백', 916),
 ('냄새', 901),
 ('필요', 893),
 ('재료', 853),
 ('트레이', 780),
 ('다양', 777),
 ('채소', 776),
 ('포장', 775),
 ('음료', 754),
 ('소재', 752),
 ('유지', 737),
 ('걱정', 704),
 ('다이소', 703),
 ('박스', 675),
 ('데비 마이어', 641),
 ('라벨', 628),
 ('소스', 627),
 ('프리', 613),
 ('생각', 597),
 ('구성', 587),
 ('구매', 587),
 ('계란', 586),
 ('편리', 581),
 ('확인', 563),
 ('냉장', 563),
 ('내용물', 561),
 ('세척', 553),
 ('깔끔', 546),
 ('이지', 539),
 ('위생', 526),
 ('살림', 517),
 ('요리', 516),
 ('마음', 510),
 ('락앤락', 504),
 ('이번', 496),
 ('전자레인지', 492),
 ('용량', 488),
 ('관리', 480),
 ('요즘', 478),
 ('재생', 478),
 ('신선', 477),
 ('아이', 

In [ ]:
# WordCloud(): 스타일(배경,글꼴), 최대단어수, 마스크이미지 등 옵션을 설정
# generate_from_frequencies(): 미리 정의된 단어의 빈도수를 이용해서 워드클라우드 이미지를 생성

wc = WordCloud(
    font_path='/home/mfdl/.local/share/fonts/나눔 글꼴/나눔고딕/NanumFontSetup_TTF_GOTHIC/NanumGothic.ttf', # malgunbd.ttf # LG PC
    background_color='white',
        
).generate_from_frequencies(dict(top_100))

plt.imshow(wc)
plt.axis('off')
# plt.show()

plt.savefig(f'./results/네이버_블로그_{keyword}.png')
plt.close()